# 02 — Memory Hierarchy and Data Movement

**Master syllabus coverage:** V2 1.2

## Why this matters

Weights, activations, and KV caches spend much of their time moving through a hierarchy. Layout, copies, allocation, and reuse can dominate mathematically identical programs.

## Learning objectives

- Explain latency, bandwidth, cache locality, and arithmetic intensity.
- Inspect contiguous and strided views in NumPy and PyTorch.
- Distinguish metadata-only views from materialized copies.
- Measure access-order and allocation-reuse effects without overstating one benchmark.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Arithmetic waits for data

The memory hierarchy moves from tiny/fast registers and caches to larger/slower main or
device memory and finally storage. **Latency** is delay for one access; **bandwidth** is
sustained bytes per second. Contiguous, predictable access improves cache lines,
prefetching, and GPU memory coalescing. Many ML operations are limited by movement rather
than arithmetic throughput.


In [ ]:
import time
import numpy as np
import torch

array = np.arange(4 * 6, dtype=np.float32).reshape(4, 6)
transposed = array.T
copied = np.ascontiguousarray(transposed)
for name, value in {"array": array, "transpose view": transposed, "contiguous copy": copied}.items():
    print(name, "shape=", value.shape, "strides(bytes)=", value.strides,
          "contiguous=", value.flags.c_contiguous, "owns data=", value.flags.owndata)
assert np.shares_memory(array, transposed)
assert not np.shares_memory(array, copied)


## 2. Views change metadata; copies move bytes

Reshape and transpose can often create a new interpretation of existing storage. A copy
allocates storage and moves every value. A later kernel may require a contiguous copy,
so a seemingly free view can defer cost rather than eliminate it. Inspect storage pointers,
strides, and allocation at operation boundaries.


In [ ]:
base = torch.arange(3 * 4 * 5).reshape(3, 4, 5)
view = base.transpose(0, 1)
materialized = view.contiguous()
print("base/view/materialized strides:", base.stride(), view.stride(), materialized.stride())
assert base.untyped_storage().data_ptr() == view.untyped_storage().data_ptr()
assert view.untyped_storage().data_ptr() != materialized.untyped_storage().data_ptr()


## 3. Access order changes locality

A C-contiguous matrix stores each row adjacently. Traversing the last axis in the inner
loop visits neighboring values; column-first scalar traversal jumps by a row stride.
Python overhead limits this demonstration, but the access-order principle persists in
compiled kernels where it can determine cache and memory efficiency.


In [ ]:
matrix = np.random.default_rng(42).normal(size=(384, 384)).astype(np.float32)

def scalar_sum_rows(value):
    total = 0.0
    for row in range(value.shape[0]):
        for column in range(value.shape[1]):
            total += value[row, column]
    return total

def scalar_sum_columns(value):
    total = 0.0
    for column in range(value.shape[1]):
        for row in range(value.shape[0]):
            total += value[row, column]
    return total

timings = {}
for name, function in (("row-major", scalar_sum_rows), ("column-major", scalar_sum_columns)):
    start = time.perf_counter(); total = function(matrix); timings[name] = time.perf_counter() - start
    print(name, f"{timings[name]*1e3:.1f} ms", "sum=", total)
row_sum = scalar_sum_rows(matrix)
column_sum = scalar_sum_columns(matrix)
print("rounding difference from accumulation order:", abs(float(row_sum - column_sum)))
np.testing.assert_allclose(row_sum, column_sum, rtol=1e-5, atol=1e-4)


## 4. Allocation versus reuse

Allocation requests storage and can trigger bookkeeping, synchronization, or garbage
collection. Reusing an output buffer reduces allocation traffic and peak live memory.
Reuse is valuable only when ownership and lifetimes remain clear—incorrect aliasing can
silently overwrite needed activations.


In [ ]:
left = np.ones(500_000, dtype=np.float32)
right = np.ones_like(left)
output = np.empty_like(left)

def measure(function, repeats=100):
    start = time.perf_counter()
    for _ in range(repeats): function()
    return time.perf_counter() - start

allocate_time = measure(lambda: left + right)
reuse_time = measure(lambda: np.add(left, right, out=output))
print(f"allocate each time={allocate_time:.4f}s, reuse output={reuse_time:.4f}s")
np.testing.assert_array_equal(output, left + right)


## 5. Arithmetic intensity gives a performance hypothesis

Arithmetic intensity is operations divided by bytes moved. Low-intensity operations such
as elementwise addition often become bandwidth-bound. Large matrix multiplication reuses
tiles many times and can become compute-bound. The roofline idea is a hypothesis tool:
attainable performance is limited by the lower of peak compute and bandwidth × intensity.


In [ ]:
# Approximate float32 vector add: one addition, two reads, one write.
vector_add_intensity = 1 / (3 * 4)
# Approximate n×n matmul: 2n³ operations, read A/B and write C once (optimistic).
for n in (32, 256, 2048):
    matmul_intensity = 2 * n**3 / (3 * n**2 * 4)
    print(f"n={n:4}: vector-add={vector_add_intensity:.3f} op/byte, matmul≈{matmul_intensity:.1f} op/byte")


## Failure modes to recognize

- **Hidden materialization:** a layout conversion increases latency and peak memory.
- **Poor locality:** workers wait for scattered memory despite low arithmetic count.
- **Allocation churn:** temporary buffers dominate a repeated small workload.
- **Unsafe reuse:** an output buffer overwrites a tensor still needed later.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Benchmark reduction on a contiguous tensor, its transpose, and a contiguous transpose copy.
2. Measure copy time and effective bandwidth for three buffer sizes.
3. Classify embedding lookup, LayerNorm, and large matmul as likely bandwidth- or compute-sensitive and defend each hypothesis.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can explain a speed difference using layout, movement, reuse, and arithmetic intensity rather than operation count alone.

**Next:** 03 — Numbers inside computers.
